In [ ]:
from pyspark.sql.functions import *
import json
env = dbutils.widgets.get("env").lower()
jobRunId = dbutils.widgets.get("jobRunId")
taskRunId = dbutils.widgets.get("taskRunId")
workflowId = dbutils.widgets.get("workflowId")

In [ ]:
# ---------------------------------------------------
# READ MASTER CONTROL (BRONZE CONFIG)
# ---------------------------------------------------
control = (
    spark.table("master_control")
    .filter(f"workflowId = '{workflowId}' AND layer = 'BRONZE'")
    .first()
)

cfg = json.loads(control.colMapping)

source_path = control.sourceTable        # Volume path
target_table = control.targetTable
load_type = control.loadType
archive_flag = control.archive

columns = cfg["columns"]
archive_path = cfg.get("archivePath", None)
# ---------------------------------------------------
# READ FILES FROM VOLUME
# ---------------------------------------------------
df = (
    spark.read
    .option("header", True)
    .csv(source_path)
)

# ---------------------------------------------------
# APPLY COLUMN MAPPING (rename + cast)
# ---------------------------------------------------
for c in columns:
    df = df.withColumnRenamed(c["sourceCol"], c["targetCol"])

for c in columns:
    df = df.withColumn(
        c["targetCol"],
        col(c["targetCol"]).cast(c["dtype"])
    )

# ---------------------------------------------------
# ADD AUDIT COLUMNS
# ---------------------------------------------------
df = (
    df.withColumn("_ingestion_ts", current_timestamp())
      .withColumn("_job_run_id", lit(jobRunId))
      .withColumn("_source_path", lit(source_path))
)

# ---------------------------------------------------
# WRITE TO BRONZE BASED ON LOAD TYPE
# ---------------------------------------------------
if load_type.upper() == "APPEND":

    df.write.format("delta") \
        .mode("append") \
        .saveAsTable(target_table)

elif load_type.upper() == "OVERWRITE":

    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)

else:
    raise Exception(f"Unsupported load type: {load_type}")

if archive_flag and archive_path:

    files = dbutils.fs.ls(source_path)

    for f in files:

        source_file = f.path
        file_name = source_file.split("/")[-1]

        dest_file = archive_path.rstrip("/") + "/" + file_name

        dbutils.fs.mv(source_file, dest_file)

    print(f"Archived {len(files)} files to {archive_path}")